# Vectorizar el corpus del BCV con embeddings de OpenAI

Este cuaderno genera los **vectores del corpus jurídico** para activar la búsqueda
semántica en el portal desplegado en Vercel.

**Por qué desde Colab:** algunos proveedores (OpenAI entre ellos) bloquean por país.
Colab sale a internet desde centros de datos en regiones admitidas, así que la
llamada funciona. El portal, además, ejecuta la consulta desde los servidores de
Vercel, no desde tu conexión.

**Qué produce:** `web/api/_data/vectors.f32` y `embeddings_meta.json`.
El orden de los vectores coincide con el de `corpus.json`, que es lo que hace que
la similitud coseno apunte al artículo correcto.

> Coste: ~2 000 fragmentos ≈ 1 millón de tokens ≈ unos pocos céntimos.
> Solo se ejecuta al cambiar el corpus.

## 1. Traer el repositorio

El script usa **solo la biblioteca estándar de Python**, así que no hay que instalar nada.

In [ ]:
REPO = "https://github.com/uptaragua-oficial/bcv-conocimiento-vectorial.git"

!git clone --depth 1 {REPO} 2>/dev/null || (cd bcv-conocimiento-vectorial && git pull)
%cd bcv-conocimiento-vectorial
!ls web/api/_data/

## 2. Introducir la clave de OpenAI

`getpass` evita que la clave quede visible en la salida del cuaderno.
Puedes crear una en <https://platform.openai.com/api-keys>.

In [ ]:
import os
from getpass import getpass

os.environ["EMBEDDINGS_API_KEY"] = getpass("Clave de OpenAI (sk-...): ")

# Modelo a usar. text-embedding-3-small (1536 dims) es económico y suficiente.
# Para más calidad: text-embedding-3-large (3072 dims, el archivo pasa de ~26 MB).
os.environ["EMBEDDINGS_MODEL"] = "text-embedding-3-small"

print("Modelo:", os.environ["EMBEDDINGS_MODEL"])
print("Clave cargada:", bool(os.environ["EMBEDDINGS_API_KEY"]))

## 3. Vectorizar el corpus

Verás el avance por lotes. Tarda un par de minutos.

In [ ]:
!python -m scripts.export_openai_embeddings

## 4. Comprobar que la búsqueda semántica tiene sentido

Vectoriza una consulta con el mismo modelo y muestra los fragmentos más cercanos.
Sirve para **comparar** con otros modelos: mira si los artículos que devuelve son
pertinentes para la pregunta.

In [ ]:
import json, urllib.request
import numpy as np

BASE = "web/api/_data"
meta = json.load(open(f"{BASE}/embeddings_meta.json"))
docs = json.load(open(f"{BASE}/corpus.json"))["docs"]
vecs = np.fromfile(f"{BASE}/vectors.f32", dtype="<f4").reshape(meta["n"], meta["dim"])
print(f"{meta['n']} vectores de {meta['dim']} dims · modelo {meta['modelo']}")

def embeber(texto):
    payload = json.dumps({"model": meta["modelo"], "input": [texto]}).encode()
    req = urllib.request.Request(
        meta["url"], data=payload, method="POST",
        headers={"Authorization": f"Bearer {os.environ['EMBEDDINGS_API_KEY']}",
                 "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=60) as r:
        v = np.array(json.loads(r.read())["data"][0]["embedding"], dtype="<f4")
    return v / (np.linalg.norm(v) or 1.0)

CONSULTAS = [
    "¿Qué requisitos se exigen para actuar como operador cambiario autorizado?",
    "¿Cómo se determina el tipo de cambio oficial del bolívar?",
]

for q in CONSULTAS:
    sims = vecs @ embeber(q)
    top = np.argsort(-sims)[:4]
    print(f"\n=== {q}")
    for i in top:
        d = docs[int(i)]
        print(f"  cos={sims[i]:.3f} | {d[3]} | {d[2]} | {d[6][:80].replace(chr(10),' ')}…")

## 5. Descargar los archivos generados

Guárdalos y cópialos al repositorio local, en `web/api/_data/`.
Después, en Vercel → *Settings → Environment Variables*:

| Variable | Valor |
|---|---|
| `EMBEDDINGS_API_KEY` | tu clave de OpenAI |
| `EMBEDDINGS_MODEL` | el mismo que usaste aquí |

Redespliega y comprueba que `/api/health` responde `"recuperacion": "hibrida"`.

In [ ]:
from google.colab import files

files.download("web/api/_data/vectors.f32")
files.download("web/api/_data/embeddings_meta.json")